1. Configuration du projet
2. Chargement des données
3. Exploration du corpus
4. Nettoyage
5. Chunking
6. Golden set
7. BM25
8. Dense retrieval
9. Recherche hybride
10. Reranking
11. RAG
12. Outils HLM
13. Agent
14. Évaluation
15. Analyse des erreurs

In [1]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os

PROJECT_PATH = "/content/drive/MyDrive/Assistant_HLM_RAG"

os.makedirs(PROJECT_PATH, exist_ok=True)

print(PROJECT_PATH)

/content/drive/MyDrive/Assistant_HLM_RAG


In [3]:
import os

folders = [
    "data",
    "data/raw",
    "data/raw/service_public",
    "data/raw/legi",
    "data/raw/rpls",
    "data/raw/dvf",
    "data/raw/plafonds",
    "data/processed",
    "data/indexes",
    "eval",
    "results"
]

for folder in folders:
    path = os.path.join(PROJECT_PATH, folder)
    os.makedirs(path, exist_ok=True)

print("Arborescence créée.")

Arborescence créée.


In [4]:
!pip install -q pandas numpy matplotlib tqdm

In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path
from tqdm.auto import tqdm

# Assistant HLM — RAG et agent sur données ouvertes

## Objectif

Construire un assistant permettant à un particulier :

1. de comprendre ses droits et démarches concernant le logement social ;
2. de vérifier son éligibilité à un logement social ;
3. de consulter des statistiques sur le parc social d'une commune ;
4. de consulter des informations sur les prix immobiliers d'une commune.

## Architecture envisagée

- RAG documentaire :
  - Service-Public Particuliers
  - Code de la construction et de l'habitation (LEGI)

- Outils structurés :
  - RPLS
  - DVF
  - plafonds PLAI / PLUS / PLS

- Retrieval :
  - BM25
  - Dense retrieval
  - Hybrid retrieval
  - Reranking

- Génération :
  - modèle local

- Évaluation :
  - golden set d'environ 40 questions

In [6]:
sources = pd.DataFrame([
    {
        "source": "Service-Public",
        "type": "Texte",
        "role": "RAG",
        "usage": "Démarches, demande HLM, DALO, APL, SLS, bail"
    },
    {
        "source": "LEGI - Code construction et habitation",
        "type": "Texte juridique",
        "role": "RAG",
        "usage": "Articles relatifs au logement social et à son attribution"
    },
    {
        "source": "RPLS",
        "type": "Données tabulaires",
        "role": "Outil",
        "usage": "Nombre de logements sociaux, vacance, occupation, DPE"
    },
    {
        "source": "DVF",
        "type": "Données tabulaires",
        "role": "Outil",
        "usage": "Prix de vente immobiliers par commune"
    },
    {
        "source": "Plafonds HLM",
        "type": "Barème",
        "role": "Outil",
        "usage": "Calcul d'éligibilité PLAI / PLUS / PLS"
    }
])

sources

,source,type,role,usage
0,Service-Public,Texte,RAG,"Démarches, demande HLM, DALO, APL, SLS, bail"
1,LEGI - Code construction et habitation,Texte juridique,RAG,Articles relatifs au logement social et à son ...
2,RPLS,Données tabulaires,Outil,"Nombre de logements sociaux, vacance, occupati..."
3,DVF,Données tabulaires,Outil,Prix de vente immobiliers par commune
4,Plafonds HLM,Barème,Outil,Calcul d'éligibilité PLAI / PLUS / PLS


Données
   ↓
Nettoyage
   ↓
Chunks
   ↓
Moteur de recherche
   ↓
Évaluation du moteur
   ↓
LLM
   ↓
Agent

In [7]:
golden_questions = pd.DataFrame([
    {
        "id": "q001",
        "type": "rag",
        "question": "Comment déposer une demande de logement social ?",
        "answer": "",
        "gold_docs": []
    },
    {
        "id": "q002",
        "type": "rag",
        "question": "Qu'est-ce que le droit au logement opposable (DALO) ?",
        "answer": "",
        "gold_docs": []
    },
    {
        "id": "q003",
        "type": "eligibility",
        "question": "Avec 28 000 euros de revenus pour un couple avec un enfant à Lyon, suis-je éligible à un logement social ?",
        "answer": "",
        "gold_docs": []
    },
    {
        "id": "q004",
        "type": "rpls",
        "question": "Combien de logements sociaux vacants y a-t-il à Créteil ?",
        "answer": "",
        "gold_docs": []
    },
    {
        "id": "q005",
        "type": "out_of_scope",
        "question": "Quels logements sociaux sont disponibles aujourd'hui à Créteil ?",
        "answer": "Le système ne dispose pas de données en temps réel sur les logements disponibles.",
        "gold_docs": []
    }
])

golden_questions

,id,type,question,answer,gold_docs
0,q001,rag,Comment déposer une demande de logement social ?,,[]
1,q002,rag,Qu'est-ce que le droit au logement opposable (...,,[]
2,q003,eligibility,Avec 28 000 euros de revenus pour un couple av...,,[]
3,q004,rpls,Combien de logements sociaux vacants y a-t-il ...,,[]
4,q005,out_of_scope,Quels logements sociaux sont disponibles aujou...,Le système ne dispose pas de données en temps ...,[]


In [8]:
golden_path = os.path.join(
    PROJECT_PATH,
    "eval",
    "golden_draft.jsonl"
)

golden_questions.to_json(
    golden_path,
    orient="records",
    lines=True,
    force_ascii=False
)

print("Sauvegardé :", golden_path)

Sauvegardé : /content/drive/MyDrive/Assistant_HLM_RAG/eval/golden_draft.jsonl


In [9]:
with open(golden_path, "r", encoding="utf-8") as f:
    print(f.read())

{"id":"q001","type":"rag","question":"Comment déposer une demande de logement social ?","answer":"","gold_docs":[]}
{"id":"q002","type":"rag","question":"Qu'est-ce que le droit au logement opposable (DALO) ?","answer":"","gold_docs":[]}
{"id":"q003","type":"eligibility","question":"Avec 28 000 euros de revenus pour un couple avec un enfant à Lyon, suis-je éligible à un logement social ?","answer":"","gold_docs":[]}
{"id":"q004","type":"rpls","question":"Combien de logements sociaux vacants y a-t-il à Créteil ?","answer":"","gold_docs":[]}
{"id":"q005","type":"out_of_scope","question":"Quels logements sociaux sont disponibles aujourd'hui à Créteil ?","answer":"Le système ne dispose pas de données en temps réel sur les logements disponibles.","gold_docs":[]}



| Type | Nombre approximatif |
|---|---:|
| Questions RAG Service-Public / LEGI | 15 |
| Éligibilité | 8 |
| RPLS | 5 |
| DVF | 3 |
| Hors corpus / impossibles | 3 |
| Prompt injection | 3 |
| Questions pièges / causalité | 3 |
| **Total** | **40** |

✅ Google Drive connecté

✅ dossiers créés

✅ environnement Python initialisé

✅ définition du projet

✅ liste des sources

✅ premier golden set

Importer les données

In [10]:
###import os
###import zipfile

ZIP_PATH = os.path.join(
    PROJECT_PATH,
    "data",
    "raw",
    "service_public",
    "vosdroits-latest.zip"
)

EXTRACT_PATH = os.path.join(
    PROJECT_PATH,
    "data",
    "raw",
    "service_public",
    "xml"
)

os.makedirs(EXTRACT_PATH, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH, "r") as zip_ref:
    zip_ref.extractall(EXTRACT_PATH)

print("Extraction terminée.")

KeyboardInterrupt: 

In [11]:
import os
import zipfile
from pathlib import Path

ZIP_PATH = os.path.join(
    PROJECT_PATH,
    "data",
    "raw",
    "service_public",
    "vosdroits-latest.zip"
)

# Dossier LOCAL dans Colab
EXTRACT_PATH = "/content/service_public_xml"

os.makedirs(EXTRACT_PATH, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH, "r") as zip_ref:
    zip_ref.extractall(EXTRACT_PATH)

print("Extraction terminée.")

xml_files = list(Path(EXTRACT_PATH).glob("*.xml"))

print("Nombre de fichiers XML :", len(xml_files))
print(xml_files[:5])

Extraction terminée.
Nombre de fichiers XML : 5574
[PosixPath('/content/service_public_xml/F2859.xml'), PosixPath('/content/service_public_xml/F39103.xml'), PosixPath('/content/service_public_xml/F39254.xml'), PosixPath('/content/service_public_xml/R58676.xml'), PosixPath('/content/service_public_xml/R60100.xml')]


In [12]:
from pathlib import Path

xml_files = list(Path(EXTRACT_PATH).glob("*.xml"))

print("Nombre total de fichiers XML :", len(xml_files))
print(xml_files[:10])

Nombre total de fichiers XML : 5574
[PosixPath('/content/service_public_xml/F2859.xml'), PosixPath('/content/service_public_xml/F39103.xml'), PosixPath('/content/service_public_xml/F39254.xml'), PosixPath('/content/service_public_xml/R58676.xml'), PosixPath('/content/service_public_xml/R60100.xml'), PosixPath('/content/service_public_xml/F16542.xml'), PosixPath('/content/service_public_xml/R60029.xml'), PosixPath('/content/service_public_xml/F583.xml'), PosixPath('/content/service_public_xml/R65687.xml'), PosixPath('/content/service_public_xml/F2428.xml')]


<Publication ID="F34779"
             spUrl="https://www.service-public.gouv.fr/particuliers/vosdroits/F34779">

    <dc:title>...</dc:title>

    <dc:subject>
        Papiers - Citoyenneté - Élections
    </dc:subject>

    <dc:description>...</dc:description>

    <dc:identifier>F34779</dc:identifier>

    <Theme>
        <Titre>...</Titre>
    </Theme>

    <Introduction>
        ...
    </Introduction>

    ...
</Publication>

In [13]:
import xml.etree.ElementTree as ET
import pandas as pd
from pathlib import Path

records = []

for file in Path(EXTRACT_PATH).glob("F*.xml"):
    try:
        tree = ET.parse(file)
        root = tree.getroot()

        ns = {
            "dc": "http://purl.org/dc/elements/1.1/"
        }

        doc_id = root.attrib.get("ID")
        url = root.attrib.get("spUrl")
        doc_type = root.attrib.get("type")
        last_major_update = root.attrib.get(
            "dateDerniereModificationImportante"
        )

        title_elem = root.find("dc:title", ns)
        subject_elem = root.find("dc:subject", ns)
        desc_elem = root.find("dc:description", ns)
        date_elem = root.find("dc:date", ns)

        title = (
            title_elem.text.strip()
            if title_elem is not None and title_elem.text
            else None
        )

        subject = (
            subject_elem.text.strip()
            if subject_elem is not None and subject_elem.text
            else None
        )

        description = (
            desc_elem.text.strip()
            if desc_elem is not None and desc_elem.text
            else None
        )

        date = (
            date_elem.text.strip()
            if date_elem is not None and date_elem.text
            else None
        )

        records.append({
            "document_id": doc_id,
            "title": title,
            "subject": subject,
            "description": description,
            "type": doc_type,
            "date": date,
            "last_major_update": last_major_update,
            "url": url,
            "file": str(file)
        })

    except Exception as e:
        print("Erreur :", file, e)

# **Vérifier** les thèmes disponibles *texte en italique*

In [14]:
df_service_public = pd.DataFrame(records)

print(df_service_public.shape)
df_service_public.head()

(3004, 9)


,document_id,title,subject,description,type,date,last_major_update,url,file
0,F2859,Comment la journée de solidarité est-elle acco...,Travail - Formation,La journée de solidarité prend la forme d'une ...,Fiche Question-réponse conditionnée,modified 2025-10-20,2023-01-01T00:00:00,https://www.service-public.gouv.fr/particulier...,/content/service_public_xml/F2859.xml
1,F39103,Qu'est-ce qu'un véhicule terrestre à moteur ?,Argent - Impôts - Consommation,Un véhicule terrestre à moteur (VTM) est un en...,Fiche Question-réponse conditionnée,modified 2025-05-28,None,https://www.service-public.gouv.fr/particulier...,/content/service_public_xml/F39103.xml
2,F39254,Dispositifs de pré-enregistrement du système e...,Étranger - Europe,Un dispositif de pré-enregistrement de type ki...,Fiche Question-réponse conditionnée,modified 2026-04-10,None,https://www.service-public.gouv.fr/particulier...,/content/service_public_xml/F39254.xml
3,F16542,Qu'est-ce que le certificat provisoire d'immat...,Transports - Mobilité,Un certificat provisoire d’immatriculation (CP...,Fiche Question-réponse conditionnée,modified 2026-01-01,2026-01-01T00:00:00,https://www.service-public.gouv.fr/particulier...,/content/service_public_xml/F16542.xml
4,F583,Congé de paternité et d'accueil de l'enfant da...,Travail - Formation,"L’agent public, second parent d'un enfant (pèr...",Fiche d'information conditionnée,modified 2026-06-01,2026-06-01T00:00:00,https://www.service-public.gouv.fr/particulier...,/content/service_public_xml/F583.xml


In [15]:
logement_subjects = df_service_public[
    df_service_public["subject"]
    .fillna("")
    .str.contains("logement", case=False)
]

print(logement_subjects.shape)

logement_subjects[
    ["document_id", "title", "subject"]
].head(50)

(350, 9)


,document_id,title,subject
8,F39743,Vente en l’état futur d’achèvement (Vefa),Logement
9,F282,Peut-on acheter son logement social ?,Logement
16,F1550,Quel mandat de vente peut-on confier à un agen...,Logement
21,F36762,Quelle autorisation d'urbanisme déposer pour c...,Logement
28,F38659,Pays Basque : montant du loyer pour un bail d'...,Logement
36,F640,Ravalement de façade d'un immeuble : à quelle ...,Logement
48,F31601,Peut-on conserver son logement social en cas d...,Logement
60,F31858,Peut-on brûler des déchets verts (végétaux) da...,Logement
62,F33173,Aide au logement : qu'est-ce que l'aide Mobili...,Logement
78,F36956,Peut-on reconstruire un bâtiment après sa dest...,Logement


# ***Puis rechercher spécifiquement le logement social***

In [16]:
df_service_public["search_text"] = (
    df_service_public["title"].fillna("")
    + " "
    + df_service_public["description"].fillna("")
    + " "
    + df_service_public["subject"].fillna("")
)

In [17]:
keywords = [
    "logement social",
    "HLM",
    "habitation à loyer modéré",
    "DALO",
    "droit au logement opposable",
    "demande de logement social",
    "surloyer",
    "supplément de loyer de solidarité",
    "APL",
    "aide personnalisée au logement"
]

pattern = "|".join(keywords)

df_hlm_candidates = df_service_public[
    df_service_public["search_text"]
    .str.contains(pattern, case=False, regex=True)
].copy()

print("Nombre de fiches candidates :", len(df_hlm_candidates))

Nombre de fiches candidates : 24


In [18]:
df_hlm_candidates[
    ["document_id", "title", "subject", "description"]
].sort_values("title")

,document_id,title,subject,description
1544,F12006,Aide personnalisée au logement (APL),Logement,"L'APL peut être versée, sous certaines conditi..."
2552,F1280,Allocation de logement sociale (ALS),Logement,L'ALS est attribuée sous conditions. L'ALS est...
2787,F37237,Dalo,Logement,Dalo
1723,F18005,Droit au logement opposable (Dalo) : faire val...,Logement,En cas de demande de logement social restée in...
733,F35683,Faire une demande de logement social,Logement,Faire une demande de logement social
127,F10007,Faire une demande de logement social (HLM),Logement,"Pour obtenir un logement social, vous devez fa..."
176,F35381,L'activité partielle de longue durée (APLD) ex...,"Social - Santé, Ressources humaines",L'activité partielle de longue durée (APLD) pe...
1285,F32037,"L'aide personnelle au logement (APL, ALF ou AL...",Logement,"En cas de loyers impayés, l'aide personnelle a..."
371,F2559,Logement social : le bailleur peut-il résilier...,Logement,Le bailleur social peut résilier le bail du lo...
2301,F21051,Logement social : supplément de loyer de solid...,Logement,Un surloyer est réclamé quand les revenus des ...


Étape suivante : extraire le texte complet

In [19]:
# 1
df_service_public.shape

(3004, 10)

In [20]:
# 2
df_service_public["subject"].value_counts()

,count
subject,
Travail - Formation,561
Argent - Impôts - Consommation,383
Famille - Scolarité,374
Logement,333
Social - Santé,254
Justice,232
Transports - Mobilité,213
Papiers - Citoyenneté - Élections,197
Étranger - Europe,170


In [21]:
# 3
logement_subjects[
    ["document_id", "title", "subject"]
].head(50)


,document_id,title,subject
8,F39743,Vente en l’état futur d’achèvement (Vefa),Logement
9,F282,Peut-on acheter son logement social ?,Logement
16,F1550,Quel mandat de vente peut-on confier à un agen...,Logement
21,F36762,Quelle autorisation d'urbanisme déposer pour c...,Logement
28,F38659,Pays Basque : montant du loyer pour un bail d'...,Logement
36,F640,Ravalement de façade d'un immeuble : à quelle ...,Logement
48,F31601,Peut-on conserver son logement social en cas d...,Logement
60,F31858,Peut-on brûler des déchets verts (végétaux) da...,Logement
62,F33173,Aide au logement : qu'est-ce que l'aide Mobili...,Logement
78,F36956,Peut-on reconstruire un bâtiment après sa dest...,Logement


In [22]:
# 4
print("Nombre de fiches HLM candidates :", len(df_hlm_candidates))

df_hlm_candidates[
    ["document_id", "title", "subject"]
].sort_values("title")

Nombre de fiches HLM candidates : 24


,document_id,title,subject
1544,F12006,Aide personnalisée au logement (APL),Logement
2552,F1280,Allocation de logement sociale (ALS),Logement
2787,F37237,Dalo,Logement
1723,F18005,Droit au logement opposable (Dalo) : faire val...,Logement
733,F35683,Faire une demande de logement social,Logement
127,F10007,Faire une demande de logement social (HLM),Logement
176,F35381,L'activité partielle de longue durée (APLD) ex...,"Social - Santé, Ressources humaines"
1285,F32037,"L'aide personnelle au logement (APL, ALF ou AL...",Logement
371,F2559,Logement social : le bailleur peut-il résilier...,Logement
2301,F21051,Logement social : supplément de loyer de solid...,Logement


In [23]:
df_logement = df_service_public[
    df_service_public["subject"]
    .fillna("")
    .str.contains("Logement", case=False)
].copy()

print("Nombre total de fiches du thème Logement :", len(df_logement))

df_logement[
    ["document_id", "title", "subject"]
].sort_values("title")

Nombre total de fiches du thème Logement : 350


,document_id,title,subject
1250,F37190,Achat d'un logement en copropriété,Logement
2693,F2762,Achat ou vente en viager : quelles sont les rè...,Logement
2497,F2961,Acte de vente d'un logement en l'état futur d'...,Logement
1686,F2962,Acte de vente d'un logement existant,Logement
1850,F12702,Acte de vente d'un terrain situé dans un lotis...,Logement
...,...,...,...
1980,F32058,À quoi sert l'immatriculation des copropriétés ?,Logement
1427,F19905,Éco-prêt à taux zéro (éco-PTZ),Logement
1862,F38064,Éco-prêt à taux zéro (éco-PTZ) Copropriétés,Logement
2714,F31270,État des lieux d'entrée dans un bail d'habitation,Logement


In [24]:
pd.set_option("display.max_rows", None)

df_logement[
    ["document_id", "title"]
].sort_values("title")

,document_id,title
1250,F37190,Achat d'un logement en copropriété
2693,F2762,Achat ou vente en viager : quelles sont les rè...
2497,F2961,Acte de vente d'un logement en l'état futur d'...
1686,F2962,Acte de vente d'un logement existant
1850,F12702,Acte de vente d'un terrain situé dans un lotis...
2606,F258,Acte de vente d'un terrain à bâtir isolé
2366,F1988,Affichage de l'autorisation d'urbanisme sur le...
62,F33173,Aide au logement : qu'est-ce que l'aide Mobili...
1884,F39682,Aide au retrait-gonflement des argiles (RGA) p...
1357,F1702,Aide aux travaux d'insonorisation d'un logemen...


In [25]:
print("Total Service-Public :", len(df_service_public))
print("Thème Logement :", len(df_logement))
print("Candidates HLM strict :", len(df_hlm_candidates))

Total Service-Public : 3004
Thème Logement : 350
Candidates HLM strict : 24


In [26]:
housing_keywords = [
    "logement",
    "HLM",
    "logement social",
    "location",
    "locataire",
    "bail",
    "loyer",
    "APL",
    "ALS",
    "ALF",
    "DALO",
    "surloyer",
    "supplément de loyer",
    "expulsion",
    "dépôt de garantie",
    "état des lieux",
    "charges locatives",
    "aide au logement",
    "bailleur",
    "habitation"
]

pattern_housing = "|".join(housing_keywords)

df_housing_candidates = df_service_public[
    df_service_public["search_text"]
    .str.contains(pattern_housing, case=False, regex=True)
].copy()

print("Nombre de fiches logement élargi :", len(df_housing_candidates))

Nombre de fiches logement élargi : 505


In [27]:
df_housing_candidates[
    ["document_id", "title", "subject"]
].sort_values("title")

,document_id,title,subject
2570,F35175,A t-on droit au PTZ quand on est déjà propriét...,Argent - Impôts - Consommation
762,F1529,A-t-on droit au PTZ quand on est déjà propriét...,Argent - Impôts - Consommation
1250,F37190,Achat d'un logement en copropriété,Logement
2693,F2762,Achat ou vente en viager : quelles sont les rè...,Logement
2497,F2961,Acte de vente d'un logement en l'état futur d'...,Logement
1686,F2962,Acte de vente d'un logement existant,Logement
1850,F12702,Acte de vente d'un terrain situé dans un lotis...,Logement
2606,F258,Acte de vente d'un terrain à bâtir isolé,Logement
2366,F1988,Affichage de l'autorisation d'urbanisme sur le...,Logement
62,F33173,Aide au logement : qu'est-ce que l'aide Mobili...,Logement


In [28]:
df_corpus_service_public = df_housing_candidates[
    df_housing_candidates["subject"]
    .fillna("")
    .str.contains("Logement", case=False)
].copy()

print("Corpus Service-Public retenu :", len(df_corpus_service_public))

Corpus Service-Public retenu : 350


In [29]:
df_corpus_service_public[
    ["document_id", "title", "subject"]
].sort_values("title")

,document_id,title,subject
1250,F37190,Achat d'un logement en copropriété,Logement
2693,F2762,Achat ou vente en viager : quelles sont les rè...,Logement
2497,F2961,Acte de vente d'un logement en l'état futur d'...,Logement
1686,F2962,Acte de vente d'un logement existant,Logement
1850,F12702,Acte de vente d'un terrain situé dans un lotis...,Logement
2606,F258,Acte de vente d'un terrain à bâtir isolé,Logement
2366,F1988,Affichage de l'autorisation d'urbanisme sur le...,Logement
62,F33173,Aide au logement : qu'est-ce que l'aide Mobili...,Logement
1884,F39682,Aide au retrait-gonflement des argiles (RGA) p...,Logement
1357,F1702,Aide aux travaux d'insonorisation d'un logemen...,Logement


In [30]:
CORPUS_META_PATH = os.path.join(
    PROJECT_PATH,
    "data",
    "processed",
    "service_public_logement_metadata.csv"
)

df_corpus_service_public.to_csv(
    CORPUS_META_PATH,
    index=False,
    encoding="utf-8-sig"
)

print("Sauvegardé :", CORPUS_META_PATH)

Sauvegardé : /content/drive/MyDrive/Assistant_HLM_RAG/data/processed/service_public_logement_metadata.csv
